# Using OpenAI models on Amazon Bedrock with Strands Agents

## Overview

OpenAI's GPT-5 models are available on Amazon Bedrock, callable through the [OpenAI Responses API](https://strandsagents.com/docs/user-guide/concepts/model-providers/openai-responses/). Strands' `OpenAIResponsesModel` provider talks to that API, so you can run an agent against an OpenAI model using AWS-native authentication and without a separate OpenAI account.

In this example we use `openai.gpt-5.6-terra` on Amazon Bedrock as the agent's model, with a simple `current_time` and `current_weather` tool use case. We cover both ways to authenticate below: a Bedrock API key and a short-term generated token.

## Tutorial Details

| Information            | Details                                        |
|:-----------------------|:-----------------------------------------------|
| Agent structure        | Single agent                                   |
| Model                  | `openai.gpt-5.6-terra` (from OpenAI)                 |
| Runs on                | Amazon Bedrock                                  |
| Strands model provider | `OpenAIResponsesModel`                          |
| Custom tools           | current_time, current_weather                  |

## Architecture

<div style="text-align:center">
    <img src="images/simple_agent.png" width="65%" />
</div>

## What you'll learn
* Configure an OpenAI model on Amazon Bedrock with the `OpenAIResponsesModel` provider
* See the two things a Bedrock OpenAI call needs: the Mantle endpoint and a short-lived token
* Build a tool-using agent and run it
* Simplify the setup with the `bedrock_mantle_config` shorthand

## Setup and prerequisites

### Prerequisites
* Python 3.10+
* AWS account with AWS credentials configured (AWS CLI, environment variables, or an execution role)
* Access to OpenAI models (e.g. `openai.gpt-5.6-terra`) on Amazon Bedrock, [guide](https://docs.aws.amazon.com/bedrock/latest/userguide/model-access.html)

Install the required packages for our agent:

In [ ]:
# installing pre-requisites
!pip install -r requirements.txt

### Importing dependency packages

Now let's import the dependency packages:

In [ ]:
import os
from datetime import datetime
from datetime import timezone as tz
from typing import Any
from zoneinfo import ZoneInfo

from strands import Agent, tool
from strands.models.openai_responses import OpenAIResponsesModel

### Authenticate to Bedrock

To call an OpenAI model on Amazon Bedrock, the provider needs two things:

1. **An endpoint** to send the request to. Bedrock serves OpenAI models through **Mantle**, its OpenAI-compatible endpoint: `https://bedrock-mantle.<region>.api.aws/openai/v1`.
2. **A short-lived token** that proves you are allowed to call it.

We build both below. First, set your region. This notebook generates the token from your configured AWS credentials, so make sure they are set (AWS CLI, environment variables, or an execution role).

In [ ]:
os.environ["AWS_REGION"] = "us-east-1"
region = os.environ["AWS_REGION"]

# Optionally confirm your credentials are configured:
# !aws sts get-caller-identity

### Define the tools

Let's set up two tools for the agent to call:

In [ ]:
@tool
def current_time(timezone: str = "UTC") -> str:
    if timezone.upper() == "UTC":
        timezone_obj: Any = tz.utc
    else:
        timezone_obj = ZoneInfo(timezone)

    return datetime.now(timezone_obj).isoformat()


@tool
def current_weather(city: str) -> str:
    # Dummy implementation. Replace with actual weather API call.
    return "sunny"

### Build the model and agent

Now create the model, passing the two pieces from above explicitly through `client_args`: `base_url` is the Mantle endpoint, and `api_key` is a short-lived token that `provide_token` mints from your AWS credentials. Then build the agent with the tools.

In [ ]:
from aws_bedrock_token_generator import provide_token

model = OpenAIResponsesModel(
    model_id="openai.gpt-5.6-terra",
    client_args={
        "api_key": provide_token(region=region),  # short-lived token from your AWS credentials
        "base_url": f"https://bedrock-mantle.{region}.api.aws/openai/v1",  # the Mantle endpoint
    },
)

system_prompt = "You are a simple agent that can tell the time and the weather"
agent = Agent(model=model, system_prompt=system_prompt, tools=[current_time, current_weather])

### Run the agent

Let's ask a simple question first. The model answers directly, without needing a tool:

In [ ]:
response = agent("What is the capital of British Columbia?")

Now ask something that needs the tools. The agent calls `current_time` and `current_weather` and uses their results:

In [ ]:
response = agent("What is the time and weather in Philadelphia?")

### Inspect the response

Let's look at the usage for the last query by examining the response `metrics`:

In [ ]:
from pprint import pprint

pprint(vars(response.metrics))

### The shorthand: let the provider do it for you

You just built the endpoint and minted a token by hand. `OpenAIResponsesModel` can do both for you: pass `bedrock_mantle_config` and it builds the Mantle endpoint from your region and mints a fresh token from your AWS credentials on each request, so you write neither `base_url` nor `api_key`:

```python
model = OpenAIResponsesModel(
    bedrock_mantle_config={"region": region},
    model_id="openai.gpt-5.6-terra",
)
```

When you use `bedrock_mantle_config`, do not also pass `base_url` or `api_key` in `client_args`; the provider derives them and raises a `ValueError` if they are present. This shorthand needs `strands-agents` 1.47.0 or later, which routes GPT-5 models to the correct Mantle path. The explicit `base_url` approach above works with any version.

### Another token option: a Bedrock API key

Instead of minting a token with `provide_token`, you can generate a **Bedrock API key** in the AWS Management Console and use it as the `api_key` (with the same Mantle `base_url`). Set it as `AWS_BEARER_TOKEN_BEDROCK`, then:

```python
model = OpenAIResponsesModel(
    model_id="openai.gpt-5.6-terra",
    client_args={
        "api_key": os.environ["AWS_BEARER_TOKEN_BEDROCK"],  # your Bedrock API key
        "base_url": f"https://bedrock-mantle.{region}.api.aws/openai/v1",
    },
)
```

## Summary

In this notebook you called an OpenAI model hosted on Amazon Bedrock with the `OpenAIResponsesModel` provider and the Responses API. You saw the two things a Bedrock OpenAI call needs, the Mantle endpoint and a short-lived token, built a tool-using agent with them, ran it, and inspected its metrics. You also saw the `bedrock_mantle_config` shorthand that handles the endpoint and token for you. This completes the model providers tutorial: you have now run a Strands agent against a local Ollama model, an Azure OpenAI model through LiteLLM, and an OpenAI model on Amazon Bedrock.